In [2]:
# Cell 1: setup
using CSV
using DataFrames
using LinearAlgebra
using Zygote
using Optimisers
using Surrogates
using Random
using Optimization, OptimizationOptimJL
using Statistics
include("NRTL_model.jl")
Random.seed!(12345)

# define global constants
const ALPHA_CONST = 0.2
const MW = [18.01528, 96.084, 88.106]  # H2O, Furfural, EA
println("✅ All packages loaded and constants defined.")

✅ All packages loaded and constants defined.


In [3]:
# Cell 2: helper fctns
function mole_to_mass(x)
    w = x .* MW
    return w ./ sum(w)
end

function mass_to_mole(w)
    x = w ./ MW
    return x ./ sum(x)
end

function load_tie_lines(path::String; indices=nothing)
    df = CSV.read(path, DataFrame)
    ok_df = filter(row -> row.Status == "OK", df)
    tie_lines = []

    if isnothing(indices)
        # Load all rows
        for row in eachrow(ok_df)
            x1_mass = [row.XWIN1, row.XFIN1, row.XEAIN1]
            x2_mass = [row.XWIN2, row.XFIN2, row.XEAIN2]
            x1 = mass_to_mole(x1_mass)
            x2 = mass_to_mole(x2_mass)
            push!(tie_lines, (x1, x2))
        end
    else
        # Load only these rows
        for i in indices
            row = ok_df[i, :]
            x1 = [row.XWIN1, row.XFIN1, row.XEAIN1]
            x2 = [row.XWIN2, row.XFIN2, row.XEAIN2]
            push!(tie_lines, (x1, x2))
        end
    end

    println("Loaded $(length(tie_lines)) tie-lines from file.")
    return tie_lines
end


function reshape_tau(vec::Vector{Float64})
    τ = zeros(3, 3)
    k = 1
    for i in 1:3, j in 1:3
        if i != j
            τ[i, j] = vec[k]
            k += 1
        end
    end
    return τ
end

function loss_from_tau_vector(tau_vec, tie_lines)
    τ = reshape_tau(tau_vec)
    total_loss = 0.0
    
    for (x1_exp, x2_exp) in tie_lines
        # For each component i, equilibrium requires: γ₁ᵢ·x₁ᵢ = γ₂ᵢ·x₂ᵢ
        γ1 = activity_coefficients(x1_exp, τ)
        γ2 = activity_coefficients(x2_exp, τ)
        
        for i in 1:length(x1_exp)
            equilibrium_error = log(max(γ1[i], 1e-10)) + log(max(x1_exp[i], 1e-10)) - log(max(γ2[i], 1e-10)) - log(max(x2_exp[i], 1e-10))

            total_loss += equilibrium_error^2
        end
    end
    avg_loss = total_loss / length(tie_lines)
    return avg_loss
end

println("All helper functions defined successfully")

All helper functions defined successfully


In [4]:
# Cell 3: Load Data
tie_lines= load_tie_lines(joinpath("TrainingSets", "DATA_SALT_1_75.csv"))

println("\n All tie-lines are loaded and ready")

Loaded 305 tie-lines from file.

 All tie-lines are loaded and ready


In [5]:
# Cell 4: phase 1 of Tau search
println("Starting PHASE 1: Building surrogate model...")

lb = fill(-5, 6)  # there are 6 non-zero parameters
ub = fill(5, 6)
iter_counter = Threads.Atomic{Int}(0)

# define step limits for each phase
num_initial_samples = 8000
num_surrogate_steps = 150
total_phase1_and_2_limit = num_initial_samples + num_surrogate_steps

# Wrapper (if multithreading)
function f_wrapper_threadsafe(tau_vec_params)
    new_iter = Threads.atomic_add!(iter_counter, 1)
    
    if new_iter > total_phase1_and_2_limit
        println("! HARD STOP: Reached total limit for Phase 1+2. Forcing stop.")
        return 1e6
    end
    
    if new_iter % 5 == 0
        println("  Evaluating sample $new_iter...")
    end
    
    tau_vec_array = collect(tau_vec_params)
    
    return loss_from_tau_vector(tau_vec_array, tie_lines)
end

# generate and eval samples in parallel
println("Generating initial $(num_initial_samples) samples...")
X_tuples = sample(num_initial_samples, lb, ub, SobolSample())
Y = Vector{Float64}(undef, num_initial_samples)

using LinearAlgebra
BLAS.set_num_threads(1)

# parallel eval of initial samples
Threads.@threads for i in 1:num_initial_samples
    Y[i] = f_wrapper_threadsafe(X_tuples[i])
    
    # progress prints
    if i % 5 == 0
        println("  Completed $i/$(num_initial_samples) initial samples...")
    end
end

println("\nPHASE 1 COMPLETE. Initial map created with $(length(Y)) points.")
println("Best loss so far: $(minimum(Y))")
println("Worst loss: $(maximum(Y))")
println("Mean loss: $(mean(Y))")


Starting PHASE 1: Building surrogate model...
Generating initial 8000 samples...
  Evaluating sample 0...
  Completed 5/8000 initial samples...
  Evaluating sample 5...
  Completed 10/8000 initial samples...
  Evaluating sample 10...
  Completed 15/8000 initial samples...
  Evaluating sample 15...
  Completed 20/8000 initial samples...
  Evaluating sample 20...
  Completed 25/8000 initial samples...
  Evaluating sample 25...
  Completed 30/8000 initial samples...
  Evaluating sample 30...
  Completed 35/8000 initial samples...
  Evaluating sample 35...
  Completed 40/8000 initial samples...
  Evaluating sample 40...
  Completed 45/8000 initial samples...
  Evaluating sample 45...
  Completed 50/8000 initial samples...
  Evaluating sample 50...
  Completed 55/8000 initial samples...
  Evaluating sample 55...
  Completed 60/8000 initial samples...
  Evaluating sample 60...
  Completed 65/8000 initial samples...
  Evaluating sample 65...
  Completed 70/8000 initial samples...
  Evaluating

In [6]:
println("PHASE 2: surrogate-free optimization")

# Utility functions
sample_tie_lines(tie_lines, n) = rand(tie_lines, n)
loss_on_sample(tau_vec, full_tie_lines; n=8) = loss_from_tau_vector(tau_vec, sample_tie_lines(full_tie_lines, n))

# custom Bayesian
function custom_bayesian_optimization(objective_func, bounds_lower, bounds_upper, max_iter=1000)
    
    # track evals
    all_params = Vector{Vector{Float64}}()
    all_losses = Vector{Float64}()
    
    # start with best point from Phase 1
    best_idx_phase1 = argmin(Y)
    current_best_params = collect(X_tuples[best_idx_phase1])
    current_best_loss = objective_func(current_best_params)
    
    push!(all_params, copy(current_best_params))
    push!(all_losses, current_best_loss)
    
    # multistart local search strategy
    n_restarts = 50
    search_radius = 0.2  # use 20% of parameter range
    
    for restart in 1:n_restarts
        
        # random start near current best
        start_point = current_best_params + search_radius * (bounds_upper - bounds_lower) .* (rand(length(bounds_lower)) .- 0.5)
        start_point = max.(bounds_lower, min.(bounds_upper, start_point))  # Clamp to bounds
        
        # perform local optimization from this point
        current_point = copy(start_point)
        step_size = 0.1
        
        for local_iter in 1:20  # 20 local steps per restart
            
            # try small random perturbations
            best_local_loss = objective_func(current_point)
            best_local_point = copy(current_point)
            
            # try 8 random directions
            for direction in 1:8
                perturbation = step_size * (bounds_upper - bounds_lower) .* (rand(length(bounds_lower)) .- 0.5)
                test_point = current_point + perturbation
                test_point = max.(bounds_lower, min.(bounds_upper, test_point))  # Clamp to bounds
                
                test_loss = objective_func(test_point)
                push!(all_params, copy(test_point))
                push!(all_losses, test_loss)
                
                if test_loss < best_local_loss
                    best_local_loss = test_loss
                    best_local_point = copy(test_point)
                end
            end
            
            # move to best local point
            current_point = best_local_point
            
            # update global best
            if best_local_loss < current_best_loss
                current_best_loss = best_local_loss
                current_best_params = copy(best_local_point)
            end
            
            # adaptive step size
            step_size *= 0.95
        end
        
        # force stop if target achieved
        if current_best_loss <= 0.08
            break
        end
    end
    
    return current_best_params, current_best_loss, all_params, all_losses
end

# smart sampling obj
function smart_objective(params)
    tau_vec = collect(params)
    
    # start with 8 tie lines
    loss1 = loss_on_sample(tau_vec, tie_lines; n=8)
    
    # if looking good, validate with 10 tie lines
    if loss1 < 0.15
        loss2 = loss_on_sample(tau_vec, tie_lines; n=10)
        return 0.7 * loss1 + 0.3 * loss2  # weighted average
    end
    
    return loss1
end

# run custom optimization
println("Running custom optimization...")
start_time = time()

best_params_vector, best_loss_surrogate, all_evaluated_params, all_evaluated_losses = 
    custom_bayesian_optimization(smart_objective, lb, ub, 1000)

elapsed = round(time() - start_time, digits=1)
println("Optimization completed in $(elapsed)s")

# phase 2 validation
validation_loss = loss_on_sample(best_params_vector, tie_lines; n=10)

# Results summary
println("\n--- PHASE 2 RESULTS ---")
println("Total evaluations: $(length(all_evaluated_losses))")
println("Best parameters: $(round.(best_params_vector, digits=4))")
println("Training loss: $(round(best_loss_surrogate, digits=4))")
println("Validation loss: $(round(validation_loss, digits=4))")

# Success assessment
if validation_loss <= 0.1
    println("TARGET ACHIEVED! Loss ≤ 0.1")
elseif validation_loss <= 0.15
    println("Good progress, close to target")
else
    println("Phase 3 refinement needed")
end

# Show improvement from Phase 1
phase1_best = minimum(Y)
improvement = round((phase1_best - validation_loss) / phase1_best * 100, digits=1)
println("Improvement from Phase 1: $(improvement)%")
println("Validation loss: $(round(validation_loss, digits=4))")
println("\nReady for Phase 3 with $(best_params_vector)")

PHASE 2: surrogate-free optimization
Running custom optimization...
Optimization completed in 0.9s

--- PHASE 2 RESULTS ---
Total evaluations: 161
Best parameters: [4.4328, 4.2707, 2.7576, -1.1918, 2.9394, 1.2085]
Training loss: 0.0175
Validation loss: 0.0137
TARGET ACHIEVED! Loss ≤ 0.1
Improvement from Phase 1: 88.6%
Validation loss: 0.0137

Ready for Phase 3 with [4.432758368399273, 4.270696444127857, 2.757580993243618, -1.1918019224867393, 2.93938725659088, 1.2084555367550263]


In [7]:
# PHASE 3: improved multistart optimization

println("=== PHASE 3: IMPROVED OPTIMIZATION ===")

# set up variables
phase2_params = [1.9744, 1.9019, 2.0, 0.9979, 2.0, 0.5422]
println("Starting from Phase 2: loss ≈ 0.13")

# create bounds if missing
if !@isdefined(lb) || !@isdefined(ub)
    lb = [0.5, 0.5, 0.5, 0.5, 0.5, 0.1]
    ub = [3.0, 3.0, 3.0, 2.0, 3.0, 1.0]
    println("Using default bounds")
end

# define sample_tie_lines if missing
if !@isdefined(sample_tie_lines)
    sample_tie_lines(tie_lines, n) = rand(tie_lines, n)
end

# using adaptive sample sizes with caching
mutable struct LossCache
    cache::Dict{Vector{Float64}, Float64}
    tolerance::Float64
end

function LossCache(tol=1e-4)
    LossCache(Dict{Vector{Float64}, Float64}(), tol)
end

function cached_loss_eval(params::Vector{Float64}, tie_lines, cache::LossCache; n=15)
    for (cached_params, cached_loss) in cache.cache
        if maximum(abs.(params .- cached_params)) < cache.tolerance
            return cached_loss
        end
    end
    
    try
        # consistent sampling for better comparisons
        sample = sample_tie_lines(tie_lines, n)
        loss = loss_from_tau_vector(params, sample)
        
        if isnan(loss) || isinf(loss) || loss < 0
            return Inf
        end
        
        # Cache the result
        cache.cache[copy(params)] = loss
        return loss
    catch
        return Inf
    end
end

# IMPROVED STRATEGY 2: smarter parameter sensitivity analysis
function analyze_parameter_landscape(base_params, bounds_lower, bounds_upper)
    println("2. Analyzing parameter landscape...")
    cache = LossCache(1e-5)
    base_loss = cached_loss_eval(base_params, tie_lines, cache; n=20)
    
    sensitivities = zeros(length(base_params))
    curvatures = zeros(length(base_params))
    
    for i in 1:length(base_params)
        param_range = bounds_upper[i] - bounds_lower[i]
        step = 0.05 * param_range  # 5% step
        
        # Three-point finite difference for gradient and curvature
        params_minus = copy(base_params)
        params_plus = copy(base_params)
        
        params_minus[i] = max(bounds_lower[i], base_params[i] - step)
        params_plus[i] = min(bounds_upper[i], base_params[i] + step)
        
        loss_minus = cached_loss_eval(params_minus, tie_lines, cache; n=20)
        loss_plus = cached_loss_eval(params_plus, tie_lines, cache; n=20)
        
        if !isinf(loss_minus) && !isinf(loss_plus)
            # Gradient (sensitivity)
            gradient = (loss_plus - loss_minus) / (2 * step)
            sensitivities[i] = abs(gradient)
            
            # Curvature (second derivative approximation)
            curvature = (loss_plus - 2*base_loss + loss_minus) / (step^2)
            curvatures[i] = abs(curvature)
        end
        
        println("   τ$(i): sensitivity = $(round(sensitivities[i], digits=4)), curvature = $(round(curvatures[i], digits=4))")
    end
    
    # return indices sorted by sensitivity, but consider curvature too
    combined_score = sensitivities .+ 0.1 * curvatures
    return sortperm(combined_score, rev=true), sensitivities, curvatures
end

# IMPROVED STRATEGY 3: focused multistart w/ adaptive perturbations
function focused_multi_start(center_params, bounds_lower, bounds_upper, sensitive_order, n_starts=6)
    println("3. Focused multi-start optimization...")
    cache = LossCache(1e-5)
    
    best_overall_params = copy(center_params)
    best_overall_loss = cached_loss_eval(center_params, tie_lines, cache; n=25)
    println("   Center loss: $(round(best_overall_loss, digits=4))")
    
    # strategy: focus perturbations on most sensitive parameters
    for start in 1:n_starts
        println("   Start $(start)/$(n_starts)...")
        
        start_params = copy(center_params)
        
        # perturb top3 most sensitive parameters more aggressively
        for (rank, param_idx) in enumerate(sensitive_order[1:min(3, length(sensitive_order))])
            param_range = bounds_upper[param_idx] - bounds_lower[param_idx]
            # More perturbation for more sensitive parameters
            max_perturbation = 0.25 / sqrt(rank)  # 25%, 18%, 14% for ranks 1,2,3
            perturbation = max_perturbation * param_range * (rand() - 0.5) * 2
            
            start_params[param_idx] += perturbation
            start_params[param_idx] = max(bounds_lower[param_idx], 
                                        min(bounds_upper[param_idx], start_params[param_idx]))
        end
        
        # small perturbations for remaining parameters
        for param_idx in sensitive_order[4:end]
            param_range = bounds_upper[param_idx] - bounds_lower[param_idx]
            perturbation = 0.08 * param_range * (rand() - 0.5) * 2  # ±8%
            
            start_params[param_idx] += perturbation
            start_params[param_idx] = max(bounds_lower[param_idx], 
                                        min(bounds_upper[param_idx], start_params[param_idx]))
        end
        
        # optimize from this starting point
        final_params, final_loss = improved_local_search(start_params, bounds_lower, bounds_upper, cache)
        
        println("     Final loss: $(round(final_loss, digits=4))")
        
        if final_loss < best_overall_loss
            best_overall_loss = final_loss
            best_overall_params = copy(final_params)
            println("     ★ NEW BEST!")
        end
    end
    
    return best_overall_params, best_overall_loss
end

# IMPROVED STRATEGY 4: smarter local search with early stopping
function improved_local_search(start_params, bounds_lower, bounds_upper, cache)
    current_params = copy(start_params)
    current_loss = cached_loss_eval(current_params, tie_lines, cache; n=25)
    
    if isinf(current_loss)
        return current_params, current_loss
    end
    
    best_params = copy(current_params)
    best_loss = current_loss
    no_improvement_count = 0
    max_no_improvement = 3
    
    # adaptive step sizes -> starts larger, gets smaller
    step_factors = [0.15, 0.08, 0.04, 0.02, 0.01]  # more reasonable steps
    
    for step_factor in step_factors
        improved_this_step = false
        
        # try each parameter
        for param_idx in 1:length(current_params)
            param_range = bounds_upper[param_idx] - bounds_lower[param_idx]
            step_size = step_factor * param_range
            
            # try both directions
            for direction in [-1, 1]
                test_params = copy(best_params)
                test_params[param_idx] += direction * step_size
                test_params[param_idx] = max(bounds_lower[param_idx], 
                                           min(bounds_upper[param_idx], test_params[param_idx]))
                
                test_loss = cached_loss_eval(test_params, tie_lines, cache; n=25)
                
                if !isinf(test_loss) && test_loss < best_loss
                    improvement = (best_loss - test_loss) / best_loss
                    if improvement > 1e-6  # Only count significant improvements
                        best_loss = test_loss
                        best_params = copy(test_params)
                        improved_this_step = true
                    end
                end
            end
        end
        
        if improved_this_step
            current_params = copy(best_params)
            no_improvement_count = 0
        else
            no_improvement_count += 1
        end
        
        # early stop if no improvement
        if no_improvement_count >= max_no_improvement
            println("     Early stopping - no improvement")
            break
        end
    end
    
    return best_params, best_loss
end

# IMPROVED STRATEGY 5: smarter simulated annealing
function improved_simulated_annealing(start_params, bounds_lower, bounds_upper, sensitive_order, max_iter=60)
    println("4. Improved simulated annealing...")
    cache = LossCache(1e-5)
    
    current_params = copy(start_params)
    current_loss = cached_loss_eval(current_params, tie_lines, cache; n=25)
    
    best_params = copy(current_params)
    best_loss = current_loss
    
    initial_temp = 0.05  # lower initial temperature
    final_temp = 0.0005
    
    improvements = 0
    accepts = 0
    
    for iter in 1:max_iter
        # temp schedule
        temp = initial_temp * (final_temp/initial_temp)^(iter/max_iter)
        
        # focus on sensitive parameters (80% of the time)
        if rand() < 0.8 && length(sensitive_order) >= 3
            param_to_change = sensitive_order[rand(1:min(3, length(sensitive_order)))]
        else
            param_to_change = rand(1:length(current_params))
        end
        
        test_params = copy(current_params)
        param_range = bounds_upper[param_to_change] - bounds_lower[param_to_change]
        
        # adaptive step size based on temperature and parameter sensitivity
        base_step = temp * param_range
        step = base_step * (0.5 + rand())  # random between 0.5x and 1.5x base step
        step *= (rand() < 0.5 ? -1 : 1)  # random direction
        
        test_params[param_to_change] += step
        test_params[param_to_change] = max(bounds_lower[param_to_change], 
                                          min(bounds_upper[param_to_change], test_params[param_to_change]))
        
        test_loss = cached_loss_eval(test_params, tie_lines, cache; n=20)
        
        # accept if better, or probabilistically if worse
        delta = test_loss - current_loss
        accept = false
        
        if delta < 0
            accept = true
        elseif temp > 1e-6
            prob = exp(-delta/temp)
            accept = rand() < prob
        end
        
        if accept && !isinf(test_loss)
            current_params = copy(test_params)
            current_loss = test_loss
            accepts += 1
            
            if test_loss < best_loss
                best_loss = test_loss
                best_params = copy(test_params)
                improvements += 1
            end
        end
    end
    
    println("   SA improvements: $(improvements), acceptance rate: $(round(accepts/max_iter*100, digits=1))%")
    return best_params, best_loss
end

# execute improved optimization
println("\n1. Starting improved Phase 3...")
start_time = time()

# Step 1: smarter parameter analysis
sensitive_order, sensitivities, curvatures = analyze_parameter_landscape(phase2_params, lb, ub)
println("   Most sensitive parameters: τ$(sensitive_order[1]), τ$(sensitive_order[2]), τ$(sensitive_order[3])")

# Step 2: focused multistart optimization
multi_params, multi_loss = focused_multi_start(phase2_params, lb, ub, sensitive_order, 5)

# Step 3: improved simulated annealing
final_params, final_loss = improved_simulated_annealing(multi_params, lb, ub, sensitive_order, 50)

# Step 4: final validation with large sample
println("\n5. Final validation with 40 tie lines...")
validation_cache = LossCache(1e-6)
validation_loss = cached_loss_eval(final_params, tie_lines, validation_cache; n=40)

elapsed = round(time() - start_time, digits=1)
println("\nImproved optimization completed in $(elapsed)s")

# Results
println("\n=== IMPROVED PHASE 3 RESULTS ===")
println("Phase 2 start:     0.1312")
println("Multi-start best:  $(round(multi_loss, digits=4))")
println("Final optimized:   $(round(final_loss, digits=4))")
println("Validation (40):   $(round(validation_loss, digits=4))")
println("Final parameters:  $(round.(final_params, digits=4))")

# calculate improvements
phase2_loss = validation_loss
if validation_loss < phase2_loss
    improvement = round((phase2_loss - validation_loss) / phase2_loss * 100, digits=1)
    println("Total improvement: $(improvement)%")
else
    println("No net improvement")
end

# Success assessment
if validation_loss <= 0.05
    println("TARGET ACHIEVED! ≤ 0.05")
elseif validation_loss <= 0.08
    println("VERY close to target")
elseif validation_loss <= 0.10
    println("GOOD, solid improvement")  
else
    println("MODERATE progress made")
end

# Create output variables
best_params_vector = final_params
best_tau = Nothing

if length(final_params) == 6
    try
        best_tau = reshape_tau(final_params)
        println("Variables created: best_params_vector, best_tau")
    catch
        best_tau = reshape_tau(final_params)
        println("Variables created: best_params_vector, best_tau")
    end
end

println("\nImproved Phase 3 complete! Avg (validation) loss:")
println(validation_loss)
display(best_tau)

=== PHASE 3: IMPROVED OPTIMIZATION ===
Starting from Phase 2: loss ≈ 0.13

1. Starting improved Phase 3...
2. Analyzing parameter landscape...
   τ1: sensitivity = 2.412, curvature = 2.4502
   τ2: sensitivity = 3.6187, curvature = 3.2441
   τ3: sensitivity = 0.0684, curvature = 3.0023
   τ4: sensitivity = 0.012, curvature = 1.0642
   τ5: sensitivity = 0.1418, curvature = 0.1431
   τ6: sensitivity = 2.5113, curvature = 2.4671
   Most sensitive parameters: τ2, τ6, τ1
3. Focused multi-start optimization...
   Center loss: 6.6897
   Start 1/5...
     Final loss: 0.0159
     ★ NEW BEST!
   Start 2/5...
     Final loss: 0.0294
   Start 3/5...
     Final loss: 0.0164
   Start 4/5...
     Final loss: 0.0252
   Start 5/5...
     Final loss: 0.0294
4. Improved simulated annealing...
   SA improvements: 1, acceptance rate: 32.0%

5. Final validation with 40 tie lines...

Improved optimization completed in 1.0s

=== IMPROVED PHASE 3 RESULTS ===
Phase 2 start:     0.1312
Multi-start best:  0.0159
F

3×3 Matrix{Float64}:
 0.0      4.12335    4.30973
 2.0518   0.0       -0.552372
 2.85893  0.898107   0.0

In [8]:
# Cell 7: save outputs
println("\nSaving fitted tau matrix to 'fitted_tau_matrix.csv'...")
try
    df_tau = DataFrame(best_tau, :auto)
    CSV.write("fitted_tau_matrix_1_75.csv", df_tau)
    println("Successfully saved.")
catch e
    println("Error saving file: ", e)
end


Saving fitted tau matrix to 'fitted_tau_matrix.csv'...
Successfully saved.


In [9]:
println("="^60)
println("equilibrium-residual RMSE)")
println("="^60)


deltas_by_comp = [Float64[] for _ in 1:3]
weights_by_comp = [Float64[] for _ in 1:3]

for (x1_exp, x2_exp) in tie_lines
    γ1 = activity_coefficients(x1_exp, best_tau)
    γ2 = activity_coefficients(x2_exp, best_tau)

    for i in 1:3
        Δ = log(max(γ1[i] * x1_exp[i], 1e-10)) - log(max(γ2[i] * x2_exp[i], 1e-10))
        push!(deltas_by_comp[i], Δ)

        w = max((x1_exp[i] + x2_exp[i]) / 2, 1e-3)
        push!(weights_by_comp[i], w)
    end
end

# overall RMSE
all_deltas = vcat(deltas_by_comp...)
all_weights = vcat(weights_by_comp...)

rmse_logK = sqrt(mean(all_deltas .^ 2))
rmse_logK_w = sqrt(sum(all_weights .* (all_deltas .^ 2)) / sum(all_weights))

println("  RMSE_logK (unweighted, all components): ", round(rmse_logK, digits=6))
println("  RMSE_logK (weighted, all components):   ", round(rmse_logK_w, digits=6))
println("  Mean multiplicative deviation ≈ ", round(exp(rmse_logK) - 1, digits=4))

# per-component RMSE
comp_labels = ["Water", "Furfural", "Ethyl Acetate"]
println("\nPer-component RMSE (log γx residuals):")
for i in 1:3
    rmse_i = sqrt(mean(deltas_by_comp[i] .^ 2))
    rmse_i_w = sqrt(sum(weights_by_comp[i] .* (deltas_by_comp[i] .^ 2)) / sum(weights_by_comp[i]))
    println("  ", lpad(comp_labels[i], 15), ":  ",
            "RMSE = ", round(rmse_i, digits=6),
            "   RMSE_w = ", round(rmse_i_w, digits=6))
end

equilibrium-residual RMSE)
  RMSE_logK (unweighted, all components): 0.082263
  RMSE_logK (weighted, all components):   0.056172
  Mean multiplicative deviation ≈ 0.0857

Per-component RMSE (log γx residuals):
            Water:  RMSE = 0.039089   RMSE_w = 0.039047
         Furfural:  RMSE = 0.120059   RMSE_w = 0.118659
    Ethyl Acetate:  RMSE = 0.066026   RMSE_w = 0.066511
